# Cell-type signal propagation within each Schur cascade mode

Each Schur mode is used as a unit-amplitude initial signal. For that mode, the notebook automatically selects its five strongest cell-type loadings and follows each individual cell-type signal for up to ten timesteps. The matrix convention is $x_{t+1}=M_{ij}^T x_t$.

In [ ]:
from pathlib import Path

from IPython.display import display
from inhibitory_schur_modulation_script import load_mij_data
from schur_signal_propagation_script import (
    mode_cell_type_table, prepare_schur_model,
    propagate_all_mode_cell_types, propagate_mode_cell_types,
    plot_seed_mode_cell_types,
    save_schur_mode_outputs, schur_mode_loadings,
    summarize_schur_modes, summarize_schur_regions,
)

## Prepare all Schur modes

Spectral-radius normalization keeps ten-step discrete trajectories bounded while retaining the relative network and Schur structure.

In [ ]:
MATRIX_PATH = Path('matrices/mij_matrix.csv')
NETLIST_PATH = Path('matrices/mij_netlist.csv')
OUTPUT_DIR = Path('outputs/schur_modes')
TARGET_SPECTRAL_RADIUS = 0.95
HIGH_LOADING_THRESHOLD = 0.05

model = prepare_schur_model(
    MATRIX_PATH,
    normalization='spectral_radius',
    target_spectral_radius=TARGET_SPECTRAL_RADIUS,
)
print(f'{model.n_modes} Schur modes across {len(model.labels)} cell types')

In [ ]:
# The five cell types represented most strongly in every mode at t=0.
mode_membership = mode_cell_type_table(model, top_n=5)
mode_membership.head(15)

## Propagate every mode

`coupled=True` follows the cascade: off-diagonal entries of the Schur matrix transfer the seed mode's signal into downstream modes. Each row below remains a readout of one of the seed mode's five principal cell types.

In [ ]:
all_mode_signals = propagate_all_mode_cell_types(
    model,
    timesteps=5,
    top_n=5,
    amplitude=1.0,
    coupled=True,
)
all_mode_signals.head(15)

## Inspect one cascade mode

Change `mode_to_view` to any integer from zero through `model.n_modes - 1`. The table and plot show separate trajectories for that mode's five strongest cell types.

In [ ]:
mode_to_view = 10

one_mode = all_mode_signals.query('seed_mode == @mode_to_view')
one_mode.pivot(
    index='timestep', columns='cell_type', values='signal_real'
).round(6)

In [ ]:
plot_seed_mode_cell_types(
    all_mode_signals, mode=mode_to_view, value='signal_real'
);

## Compare cascade propagation with the isolated mode

The isolated version excludes transfer to other Schur coordinates and therefore follows $q_k\lambda_k^t$ exactly.

In [ ]:
isolated_mode = propagate_mode_cell_types(
    model, mode=mode_to_view, timesteps=10, top_n=5, coupled=False
)
isolated_mode.pivot(
    index='timestep', columns='cell_type', values='signal_real'
).round(6)

## Mode loadings and anatomical summaries

The following tables consolidate the former draft EDA. Mode indexes remain zero-based and preserve the original Schur ordering. A high-loading cell has `|Q[cell, mode]| > HIGH_LOADING_THRESHOLD`.

In [ ]:
mij = load_mij_data(MATRIX_PATH, NETLIST_PATH)
assert list(model.labels) == mij.labels

loadings = schur_mode_loadings(model, mij.ei.to_dict())
display(loadings.head(15))

### Mode and region summaries

In [ ]:
mode_summary = summarize_schur_modes(loadings, HIGH_LOADING_THRESHOLD)
region_summary = summarize_schur_regions(mode_summary)
display(mode_summary)
display(region_summary)

### Inspect or filter individual modes

In [ ]:
MODE_TO_INSPECT = 0
MIN_LOADING_TO_SHOW = 0.05

columns = [
    'loading_rank', 'cell_type', 'region', 'ei',
    'loading_real', 'loading_imag', 'loading_magnitude', 'energy_fraction',
]
display(loadings.query(
    'mode == @MODE_TO_INSPECT and loading_magnitude > @MIN_LOADING_TO_SHOW'
)[columns])

## Export propagation and reusable mode tables

This saves the propagation table, exact Schur arrays, tidy loading tables, regional summaries, and metadata under `outputs/schur_modes`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_mode_signals.to_csv(OUTPUT_DIR / 'schur_mode_cell_type_propagation.csv', index=False)
saved_to = save_schur_mode_outputs(
    model, loadings, mode_summary, region_summary, OUTPUT_DIR,
    matrix_path=MATRIX_PATH,
    target_spectral_radius=TARGET_SPECTRAL_RADIUS,
    high_loading_threshold=HIGH_LOADING_THRESHOLD,
)
print(f'Exported {len(all_mode_signals):,} propagation rows to {saved_to}')